# Orbital Solar Insight — Análise NASA POWER
**FIAP — Global Solution 2026**

Este notebook acompanha o pipeline `src/` e foi pensado para experimentação interativa. As células abaixo cobrem:

1. Aquisição (ou carregamento em cache) dos dados NASA POWER para 12 capitais brasileiras.
2. Análise exploratória — distribuições, sazonalidade, correlações.
3. **Q1** — previsão supervisionada da irradiância diária (regressão).
4. **Q4** — agrupamento não-supervisionado de cidades por perfil climático.
5. Interpretação dos resultados sob a ótica dos ODS 9, 11 e 13.

> Execute as células em ordem. Em ambiente novo, descomente a célula de aquisição inicial.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data_acquisition, data_preprocessing, eda, modeling, clustering

plt.style.use(config.PLT_STYLE)
pd.set_option('display.max_columns', 60)
print('cidades configuradas:', len(config.CITIES))

## 1. Aquisição de dados

A API NASA POWER (endpoint *temporal/daily/point*) é pública e não exige chave. Cada cidade gera um CSV em `data/raw/`. A função `download_all` é idempotente: se o arquivo já existir, ele é reutilizado.

**Período:** 2015-01-01 a 2024-12-31 (10 anos).  
**Variáveis:** irradiância global e em céu limpo, temperatura média/máx/mín, umidade, precipitação, vento, pressão, índice de claridade.

In [ ]:
csv_path = data_acquisition.download_all()  # idempotente — usa cache
df = pd.read_csv(csv_path, parse_dates=['date'])
df.head()

In [ ]:
print('Período:', df['date'].min().date(), '->', df['date'].max().date())
print('Linhas:', len(df), '| cidades:', df['city'].nunique())
print('\nSentinelas (-999) por variável:')
(df[config.NASA_PARAMETERS] == config.NASA_MISSING_SENTINEL).sum()

## 2. Análise exploratória

Investigamos: distribuição da irradiância por região (gradiente latitudinal esperado), sazonalidade mensal, correlações entre variáveis meteorológicas e qualidade do dado (missing).

Cada chamada de `eda.plot_*` salva um PNG em `outputs/figures/` e retorna o caminho.

In [ ]:
eda.plot_irradiance_distribution(df);

**Hipótese verificada:** Norte e Nordeste devem apresentar mediana maior, e Sul a menor. O Nordeste tende ainda a ter menor dispersão (céu limpo o ano todo).

In [ ]:
eda.plot_monthly_seasonality(df);

In [ ]:
eda.plot_correlation_matrix(df);

**Leitura:** a irradiância (`ALLSKY_SFC_SW_DWN`) deve correlacionar positivamente com temperatura, índice de claridade (`ALLSKY_KT`) e com a irradiância de céu limpo, e negativamente com umidade e precipitação. Isso já indica o conjunto natural de regressores.

In [ ]:
eda.plot_temperature_vs_irradiance(df);
eda.plot_long_term_trend(df);
eda.plot_missing_summary(df);

## 3. Q1 — Previsão supervisionada de irradiância diária

**Alvo:** `ALLSKY_SFC_SW_DWN` (kWh/m²/dia).  
**Features:** variáveis meteorológicas do dia + lags 1/7 + médias móveis 7/30 dias + componentes cíclicos de sazonalidade + região.  
**Divisão:** temporal — últimos 20% das datas no hold-out.  
**CV:** `TimeSeriesSplit(5)` no conjunto de treino.

Três modelos são comparados (baseline linear, Random Forest, XGBoost) — esperamos que o gradient boosting vença em RMSE devido a interações não-lineares (ex.: nebulosidade × estação).

In [ ]:
X, y, meta = data_preprocessing.build_feature_matrix(df)
X_tr, X_te, y_tr, y_te, meta_tr, meta_te = data_preprocessing.temporal_train_test_split(X, y, meta)
print(f'treino: {len(X_tr)} | teste: {len(X_te)} | features: {X.shape[1]}')

In [ ]:
results = modeling.train_and_compare(X_tr, X_te, y_tr, y_te)
metrics = modeling.metrics_table(results)
metrics

In [ ]:
best_name = metrics['RMSE teste'].idxmin()
best = results[best_name]
print('melhor modelo:', best_name)

modeling.plot_predictions(best, X_te, y_te, meta_te)
modeling.plot_feature_importance(best);

**Interpretação:** uma irradiância de céu limpo elevada (`CLRSKY_SFC_SW_DWN`) e um índice de claridade alto (`ALLSKY_KT`) são os preditores mais informativos — o que é fisicamente esperado. Lags da própria irradiância capturam persistência atmosférica de curto prazo. A componente cíclica `doy_sin/cos` carrega a sazonalidade.

Em termos práticos (ODS 9 e 11), um RMSE da ordem de 0,5 kWh/m²/dia em irradiância significa erro relativo aceitável para dimensionar sistemas fotovoltaicos urbanos — embora a aplicação real exija conversão para potência via *system performance ratio*.

## 4. Q4 — Clustering de cidades por perfil climático

Agregamos as séries diárias em **assinaturas climáticas anuais** (média de irradiância, amplitude térmica, umidade, precipitação, vento, índice de claridade). Como o dataset por cidade é pequeno (n=12), KMeans é adequado e estável.

Avaliamos k∈[2,8] via inércia (cotovelo) e silhueta. O **k final** está fixado em `config.KMEANS_DEFAULT_K`.

In [ ]:
signatures = data_preprocessing.aggregate_city_climatology(df)
signatures

In [ ]:
scores = clustering.evaluate_k_range(signatures)
clustering.plot_k_diagnostics(scores)
scores

In [ ]:
result = clustering.fit_kmeans(signatures)
print(f'k={result.k} | silhueta={result.silhouette:.3f} | calinski={result.calinski:.1f}')
result.summary[['city', 'region', 'biome', 'cluster']].sort_values('cluster')

In [ ]:
clustering.plot_pca_projection(result)
clustering.plot_cluster_heatmap(result)
clustering.cluster_profiles(result)

**Interpretação esperada (ODS 11 e 13):** os clusters devem espelhar, em parte, biomas e latitude — Amazônia úmida, Nordeste semiárido com alta irradiância, Cerrado, Sudeste/Sul temperados. Cada grupo aceita políticas públicas distintas: o Nordeste demanda gestão de calor extremo, o Sul tem viabilidade mais variável para solar e o Sudeste concentra demanda urbana sob estresse térmico crescente.

## 5. Discussão e limitações

- **Granularidade espacial.** NASA POWER é uma grade de ~0,5° — em centros urbanos, micro-clima e ilhas de calor não são resolvidos.
- **Fenômenos extremos.** Modelos treinados em médias diárias subestimam picos. Para aplicações de risco (ondas de calor, sobrecarga elétrica), é preciso pós-processar com distribuição de extremos.
- **Cobertura temporal.** ENSO afeta padrões — 10 anos cobrem ~2 ciclos; um intervalo maior reduz o viés temporal.
- **Validação externa.** Recomenda-se confrontar com INMET (estações de superfície) antes de uso operacional.

## Conexão com ODS

| ODS | Conexão direta |
|-----|----------------|
| 9 | Dimensionamento de geração solar urbana |
| 11 | Identificação de cidades vulneráveis a calor / oportunidades solares |
| 13 | Diagnóstico de variabilidade climática inter-regional |